In [0]:
import logging
from py4j.java_gateway import JavaObject  
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import concat_ws, lit, regexp_extract
from datetime import datetime, date
import re
import shutil
import os

##### 1. **Ingest** and **transform** 991 dialer data 

In [0]:
%python
dialer991Data  = spark.sql("""

WITH dialerRowData AS (
SELECT SourceSystem,
       PortfolioName,
       CDI,
       DialerKey, -- Είναι το κλειδί της κλήσης. Προσοχή, υπάρχουν ίδιες κλήσεις, DialerKeys, που αντιστοιχούν σε πολλαπλά χαρτοφυλάκια
       CAST(DateHour AS DATE) AS DateHour, -- Ορίζεται σαν την ώρα που έγινε η αντίστοιχη κλήση
       CaseCode,
       CallType,
       CampaignType,
       CampaignName,
       TalkTimeDuration,
       BusinessDate
FROM `p-cdh`.primary.dialer 
),

dialerDataWithOutboundCalls AS (
SELECT x.SourceSystem,
       x.PortfolioName,
       x.CDI,
       x.DialerKey, 
       x.DateHour, 
       x.CaseCode,
       x.CallType,
       x.CampaignType,
       x.CampaignName,
       x.TalkTimeDuration,
       x.BusinessDate,
       DATEADD(SECOND, 1, CAST(x.DateHour AS TIMESTAMP)) AS ActivityDatetime
FROM  dialerRowData AS x
WHERE BusinessDate = (SELECT MAX(BusinessDate) FROM dialerRowData)
AND DateHour       = (SELECT MAX(BusinessDate) FROM dialerRowData)
AND SourceSystem = 'cti'
AND CallType = 'OUTBOUND'  -- We keep only *OUTBOUND* dialer calls, as asked by the Business
)

SELECT CaseCode, 
       ActivityDatetime,
       Attempts,
       CONCAT('attempts:', Attempts, ' (cepal:', Attempts , ', external:', 0,')') AS ActivityComments
FROM (
SELECT CaseCode,
       ActivityDatetime,
       COUNT(DISTINCT DialerKey) AS Attempts
FROM dialerDataWithOutboundCalls
GROUP BY CaseCode, ActivityDatetime 
)

""")
dialer991Data.createOrReplaceTempView("dialer991Data")

In [0]:
%python
dialer991ExportData  = spark.sql("""
SELECT DISTINCT CONCAT(
       COALESCE(CaseCode, ' '), '|||', 
       '8795', '|',
       'dactivities', '|', 
        COALESCE(ActivityComments, ' '), '|',
        DATE_FORMAT(ActivityDatetime, 'yyyy-MM-dd HH:mm:ss'), '||||||||') AS ForText
FROM dialer991Data
""")
dialer991ExportData.createOrReplaceTempView("dialer991ExportData")

##### 2. Check whether yesterday’s data matches today’s data
##### Table naming conversions:
##### 1. **dialer991ExportData**: Today's calls - **Target table**
##### 2. **parsedLatestNonEmptyTextFile**: Yesterday calls, precisely the calls of the last Business Date - **Source Table**

###### Note: The final text file has two properties:
###### 1. Each line ends in "CR LF"
###### 2. An empty line exists at the end of each text file
###### 3. The text files are stored in the folder: **\\dwhsideapps\Share\in\Prod\991dialerexports**

##### Check out the documentation: https://cepalhellas.atlassian.net/wiki/spaces/DE/pages/1167949825/Dialer+991

In [0]:
def fetch_latest_business_date(env: str) -> date | None:

    """Fetch the Business Date of the latest & most recent dialer text file from the volume folder"""

    volume_folder = f"/Volumes/{env}-cdh-delivery/s3_external/reports/991dialerexports/"
    volume_folder_text_files = [f.name for f in dbutils.fs.ls(volume_folder) if f.size > 0]
    latest_business_date = max(
        [
            datetime.strptime(re.search(r"_(\d{8})_", text_file).group(1), '%Y%m%d').date() 
            for text_file in volume_folder_text_files  
        ], default = None)

    latest_business_date_str = (
        datetime.strftime(latest_business_date, '%Y%m%d') 
        if latest_business_date is not None 
        else latest_business_date
        )

    return latest_business_date_str

In [0]:
def load_and_parse_text_file(env: str, latest_business_date_str: str) -> DataFrame:

    """Fetch the latest text file from the volume folder and parse it"""

    volume_folder =  f"/Volumes/{env}-cdh-delivery/s3_external/reports/991dialerexports/"
    text_file_path = volume_folder + f'MASS_ACTIVITIES_{latest_business_date_str}_93.txt'
    dialer_data = spark.read.text(text_file_path)

    parsed_dialer_data = dialer_data.select(
        "value",
        regexp_extract("value", r"^(\d+)\|\|\|", 1).cast("String").alias("CaseCode"),
        regexp_extract("value", r"attempts:(\d+)", 1).cast("Int").alias("Attempts"),
        regexp_extract("value", r"\|(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\|{8}", 1).cast("Date").alias("ActivityDatetime")
    )
    return parsed_dialer_data

In [0]:
env = 'p'
latest_business_date_str = fetch_latest_business_date(env = env)
parsedLatestNonEmptyTextFile = load_and_parse_text_file(env = env, latest_business_date_str = latest_business_date_str)
parsedLatestNonEmptyTextFile.createOrReplaceTempView("parsedLatestNonEmptyTextFile")

In [0]:
%python 
todayUnequalToYesterdayCallsChecks = spark.sql("""
WITH TodayVersusYesterdayCalls AS (
SELECT x.CaseCode AS TodayCaseCode, 
       x.Attempts AS TodayAttempts, 
       y.CaseCode AS YesterdayCaseCode,
       y.Attempts AS YesterdayAttempts
FROM dialer991Data AS x
FULL OUTER JOIN parsedLatestNonEmptyTextFile AS y
ON x.CaseCode = y.CaseCode
),

TodayVersusYesterdayCallStats AS (
SELECT SUM(CASE WHEN TodayCaseCode IS NULL THEN 1 ELSE 0 END) AS TodayNullCaseCodes,
       SUM(CASE WHEN YesterdayCaseCode IS NULL THEN 1 ELSE 0 END) AS YesterdayNullCaseCodes,
       SUM(TodayAttempts) AS TodayTotalAttempts,
       SUM(YesterdayAttempts) AS YesterdayTotalAttempts
FROM TodayVersusYesterdayCalls
)

-- "False" value: means equal datasets, that is "yesterday" dataset is equal to today dataset
SELECT CASE WHEN COUNT(*) = 0 THEN True ELSE False END AS TodayUnequalToYesterdayCalls
FROM TodayVersusYesterdayCallStats
WHERE TodayNullCaseCodes = YesterdayNullCaseCodes
AND TodayTotalAttempts = YesterdayTotalAttempts
""")

todayUnequalToYesterdayCalls = todayUnequalToYesterdayCallsChecks.first()[0]

##### 3. Save the transformed 991 dialer data to Databricks volumes

In [0]:
today = datetime.today().strftime("%Y%m%d")
dialerTargetPath = f"/Volumes/{env}-cdh-delivery/s3_external/reports/991dialerexports/MASS_ACTIVITIES_{today}_93.txt"

# "Yesterday" dataset is **DIFFERENT** from today dataset
if todayUnequalToYesterdayCalls:
    rows = dialer991ExportData.collect()
    with open(dialerTargetPath, "w", encoding = "cp1253", errors = "replace", newline = "\r\n") as f:
        for row in rows:
            f.write(row['ForText'] + "\n")

# "Yesterday" dataset is **EQUAL** to today dataset
else:
    with open(dialerTargetPath, "w", encoding = "cp1253", errors = "replace", newline = "\r\n") as f:
        pass